# Counting and Vectorizing Words with sk-learn

Today, we're going to explore a measure called Term Frequency - Inverse Document Frequency (tf-idf). Tf-idf comes up a lot in text analysis projects because it’s both a corpus exploration method and a pre-processing step for many other text-mining measures and models.

The procedure was introduced in a 1972 paper by Karen Spärck Jones under the name “term specificity,” and the basic idea is this:

Instead of representing a term in a document by its raw frequency or its relative frequency (the term count divided by the document length), each term is *weighted* by dividing the term frequency by the number of documents in the corpus containing the word.

The overall effect of this weighting scheme is to avoid a common problem when conducting text analysis: the most frequently used words in any particular document are often the most frequently used words in all of the documents. Therefore, they are not particularly informative!

By contrast, terms with the highest tf-idf scores are the terms that are *distinctively* frequent in any particular document when that document is compared other documents. When you sort by tf-idf score, these distinctive terms rise to the top.

Let's start by importing the libraries we'll need for the notebook

In [ ]:
# import CountVectorizer from sk-learn
from sklearn.feature_extraction.text import CountVectorizer

# for dataframes
import pandas as pd

## Vectorize a teeny dataset

First, let's vectorize a teeny dataset we can see:

In [ ]:
# here's our dataset: the Olivia Rodrigo lines we were looking at before:
dataset = [
    'And ain’t it funny',
    'How you ran to her',
    'The second that we called it quits?',
    'And ain’t it funny',
    'How you said you were friends?',
    'Now it sure as hell don’t look like it',
]

Now our two lines to generate the document-term matrix (and we'll print the shape at the end just to check):

In [ ]:
# now instantiate the CountVectorizer object
cv=CountVectorizer()

# this steps generates document-term matrix for the doc;
# it's required before you do almost anything else
dtm=cv.fit_transform(dataset)

# this method gives us the feature names that the CountVectorizer vectorized:
features = cv.get_feature_names_out()

# print the shape
dtm.shape


Now we'll print out the features:

In [ ]:
print("All of the features in our dataset:")
print(str(features))

And their counts:

In [ ]:
# this method turns our doc-term matrix into an array that can be manipulated:
dtm_array = dtm.toarray()

# and print; each line is the feature counts of one line of the song
print(dtm_array)

In [ ]:
# here is some code that uses pandas to make the above slightly more legible
df = pd.DataFrame(data=dtm_array,columns=features)

print(df)

Make sense?

Now let's move onto our more complicated dataset--the Yelp reviews!

## Oveview of Yelp Review Data ##

For the next part of this lesson, we're going to use tf-idf to study Yelp reviews for restaurants in Atlanta.

This dataset was created in 2023 by [Naitian Zhou](https://naitian.org/) for a project involving questions of taste and authenticity in US restaurant reviews, inspired by work by [Sara Kay](https://ny.eater.com/2019/1/18/18183973/authenticity-yelp-reviews-white-supremacy-trap), [Yiwei Luo, Kristina Gligoric, and Dan Jurafsky](https://arxiv.org/pdf/2307.07645), and [Sharon Zukin, Scarlett Lindeman, and Laurie Hurson](https://journals-sagepub-com.proxy.library.emory.edu/doi/full/10.1177/1469540515611203). All found instances of racial and ethnic bias in such reviews, and one of the things we were hoping to do in our project was to see if we could replicate the findings in a more comprehensive dataset.

## Pre-processing: prepare the reviews

Tf-idf works on sets of documents--individual reviews in our case. We'll be using scikit-learn to count the words in the reviews. But before we do, we'll need to get the reviews out of a .jsonl file and into a list, with each review stored as its own string.

The reviews are stored in a .jsonl file that is zipped and stored on my Google Drive. Below is some code to get the zipped jsonl file from Google Drive, unzip it, and format the review text into a list for processing.

First, download the file:

In [ ]:
# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# then download the zip files
# atlanta
gdown.download('https://drive.google.com/uc?export=download&id=1gIm9NcoeY1gn9EQjRr2MojGRJ-fpBSqz', quiet=False)

# we'll use these next class, maybe
# san francisco
# gdown.download('https://drive.google.com/uc?export=download&id=1NU19CyDbRVDJfGwV9kpV-JDn5OLPuNlp', quiet=False)

# new york
# gdown.download('https://drive.google.com/uc?export=download&id=1tzL-k2wMqskcDUiA5a_VD7Z3ezu6GNJ3', quiet=False)

Then, unzip it:

In [ ]:
# unzip it
!unzip Atlanta-random.jsonl.zip


Then, process the data. So we can take a quick look at everything that's in the json file, we'll pull it into a dataframe first.

Note that while this code is written to process this particular dataset, you'll usually need to write some sort of file/text pre-processing code in order to use any particular library/method/tool. You'll get very familiar with writing code like this by the end of the course!

In [ ]:
# import more libraries
import os             # for directory/file manipulation
import json           # for json
# import pandas as pd   # for dataframes

# read in the file
atlanta_reviews_df = pd.read_json(path_or_buf="./Atlanta-random.jsonl", lines=True)

len(atlanta_reviews_df)


In [ ]:
# take a quick look at the top
atlanta_reviews_df.head()

Great! But what we really want is what's in the "comment" column, and in particular we want the value of the "text" key. So let's make a list with only that.

In [ ]:
# first extract the 'comment' values from the dataframe
comments = atlanta_reviews_df['comment'].tolist()

# create list to store reviews
reviews = []

# iterate through the comments and append the reviews to the list
for comment in comments:
  reviews.append(comment['text'])

# print out the first one to check
reviews[23540]

Oops! There's still some HTML in there. Let's do a quick cleaning pass.

In [ ]:
from bs4 import BeautifulSoup

# new array w/ clean text
reviews_clean = []

for review in reviews:
    soup = BeautifulSoup(review, "html.parser")
    text = soup.get_text(separator=' ')

    reviews_clean.append(text)

One last thing. Let's make a set of IDs so we can get a sense of what restaurant each review is about without having to print out the whole thing.

In [ ]:
# extract the 'business' values from the dataframe
businesses = atlanta_reviews_df['business'].tolist()

# create list to store business aliases
aliases = []

# iterate through the business and append the alias to the list
for business in businesses:
  aliases.append(business['alias'])

# extract ratings
ratings = atlanta_reviews_df['rating'].tolist()

# create list to store IDs
ids = []

# now put them all together into IDs
for i, alias in enumerate(aliases):
  id = alias + "-review" + str(i) + "-" + str(ratings[i]) + "stars"
  ids.append(id)

# print out the first one to check
ids[23540]

OK. Looks like we're finally ready to go!

## Vectorizing a dataset from a set of files

The reality is that you almost always will be vectorizing a dataset from a set of files, and not a few lines of Olivia Rodrigo lyrics that you type in by hand. This is how you'd do it with the Yelp reviews dataset:

In [ ]:
# instantiate the vectorizer, as before
cv=CountVectorizer()

# generates document-term matrix for all the docs
dtm=cv.fit_transform(reviews_clean)

# get the feature names aka terms
features = cv.get_feature_names_out()

# take a look at some features in the middle
print(features[3075:3175])

## At long last, the TF-IDF calculations!

It's only a few lines of code:

In [ ]:
# import our required library
from sklearn.feature_extraction.text import TfidfVectorizer

# to exclude stopwords, add the argument `stop_words='english'`
tfidf_vectorizer=TfidfVectorizer(stop_words='english', use_idf=True)

# send in all your docs here
tfidf_vectors=tfidf_vectorizer.fit_transform(reviews_clean)

**And we're done! 🎉 🎉 🎉**

---

Now, to explore the results...

First, let’s print the tf-idf values of the first document to see if they make sense.

We'll place the tf-idf scores from the first document into a pandas dataframe and sort the dataframe in descending order of scores.

In [ ]:
import textwrap

feature_names = tfidf_vectorizer.get_feature_names_out()

#get tfidf vector for first document
first_document_vector=tfidf_vectors[0]

# print the review text for the first doc
print(textwrap.fill(reviews_clean[0],100))

#print the scores for the first doc
df = pd.DataFrame(first_document_vector.T.todense(), index=feature_names, columns=["tfidf"])
df.sort_values(by=["tfidf"],ascending=False).head(10)

## Displaying the top terms for any particular document

Since one of the major uses of TF-IDF is to characterize the most significant words in any particular document, here's a standalone cell that will do that for the review number you select in the first line below:

In [ ]:
review_num = 33013

#get tfidf vector for another document
first_document_vector=tfidf_vectors[review_num]

# print the review text for the first doc
print(textwrap.fill(reviews_clean[review_num],100))

#print the scores for the first doc
df = pd.DataFrame(first_document_vector.T.todense(), index=feature_names, columns=["tfidf"])
df.sort_values(by=["tfidf"],ascending=False).head(10)

# Searching/sorting by tf-idf score

One question you often want to ask about tf-idf scores relates to individual words-- more specifically, which documents have the highest tf-idf scores for a specific word.  

You might want to search/sort this way if you were curious, for example, which documents were most uniquely about, say, boba:

In [ ]:
# new dataframe for id lookup
tfidf_df = pd.DataFrame(tfidf_vectors.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# add in a column for the titles of each article for future reference
tfidf_df['IDs'] = ids

In [ ]:
tfidf_slice_sorted = tfidf_df[['IDs', 'boba']].sort_values(by=['boba'], ascending=False)

# print out the top ten
print (tfidf_slice_sorted[:10])

To look for another word, you can swap out "boba" up there (twice) with whatever it is you want to look for

## Searching for multiple terms

Not too much different than above, but you don't need to include just one term as part of your slice.

In [ ]:
tfidf_slice_sorted = tfidf_df[['IDs', 'chicken', 'waffles']].sort_values(by=['chicken', 'waffles'], ascending=False)

# print out the top ten
print (tfidf_slice_sorted[:10])

OK! That's it for now!

# What to carry forward

**1. Words can become numbers.** Tokenizing into a document-term matrix is the shared foundation of topic modeling, classification, clustering and similarity.

**2. tf-idf finds distinctive words, not frequent words.** It answers the question: “what is this document about that others are not?”

**3. The ranking matters, not the score itself.** TF-IDF implementations differ. What you use is the ranking that results.







*Lauren F. Klein wrote version 1.0 of this notebook in 2019 based of tutorials by [Matthew Lavin](https://programminghistorian.org/en/lessons/analyzing-documents-with-tfidf) and [Kavita Ganesan](https://kavita-ganesan.com/tfidftransformer-tfidfvectorizer-usage-differences/#.XZVlcOdKhSw). Dan Sinykin supplemented it with material from Melanie Walsh's chapter [TF-IDF](https://melaniewalsh.github.io/Intro-Cultural-Analytics/features/Text-Analysis/TF-IDF.html) in 2020. Lauren Klein updated it again in 2021, 2022, 2024, and 2026.*

